In [0]:
# Databricks notebook source

# =============================================================================
# SMART MANUFACTURING INTELLIGENCE PLATFORM (SMIP)
#
# Notebook : 01_run_smip
#
# Description
# -----------------------------------------------------------------------------
# Generates all Manufacturing Master and Transactional datasets,
# validates the generated CSV files and synchronizes them to
# Unity Catalog Volumes.
#
# Author  : Sumanth Vempalle
# Version : 1.0.0
# =============================================================================

### Imports & Configuration 

In [0]:
from pathlib import Path
import shutil

# =============================================================================
# Master Data
# =============================================================================

from generator.master_data.generate_product_master import main as generate_products
from generator.master_data.generate_machine_layout import main as generate_factory
from generator.master_data.generate_operator_master import main as generate_operators
from generator.master_data.generate_operation_master import main as generate_operations
from generator.master_data.generate_press_program_master import main as generate_press_programs
from generator.master_data.generate_test_program_master import main as generate_test_programs
from generator.master_data.generate_tool_master import main as generate_tools

# =============================================================================
# Transactional Data
# =============================================================================

from generator.simulation.simulate_work_orders import main as generate_work_orders
from generator.simulation.generate_production_executions import main as generate_executions
from generator.simulation.generate_serial_numbers import main as generate_serial_numbers
from generator.simulation.simulate_press_operations import main as generate_press_operations
from generator.simulation.simulate_force_curves import main as generate_force_curves
from generator.simulation.simulate_testing import main as generate_test_results
from generator.simulation.simulate_packaging import main as generate_packaging
from generator.simulation.simulate_material_scan import main as generate_material_scans
from generator.simulation.simulate_operator_login import main as generate_operator_logins

from generator.configs.paths import (
    MASTER_DATA,
    TRANSACTIONAL_DATA
)

# =============================================================================
# Unity Catalog Volume
# =============================================================================

UNITY_VOLUME = Path("/Volumes/smip/bronze/source_data")

MASTER_VOLUME = UNITY_VOLUME / "master_data"

TRANSACTION_VOLUME = UNITY_VOLUME / "transactional_data"

# =============================================================================
# Generator Registry
# =============================================================================

MASTER_GENERATORS = [

    generate_products,
    generate_factory,
    generate_operators,
    generate_operations,
    generate_press_programs,
    generate_test_programs,
    generate_tools,

]

TRANSACTIONAL_GENERATORS = [

    generate_work_orders,
    generate_executions,
    generate_serial_numbers,
    generate_press_operations,
    generate_force_curves,
    generate_test_results,
    generate_packaging,
    generate_material_scans,
    generate_operator_logins,

]

print(f"Master generators        : {len(MASTER_GENERATORS)}")
print(f"Transactional generators : {len(TRANSACTIONAL_GENERATORS)}")

### Helper Functions

In [0]:
def clean_folder(folder):

    folder.mkdir(parents=True, exist_ok=True)

    for file in folder.glob("*.csv"):

        if file.is_file():

            file.unlink()


def execute(generators, title):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    for job in generators:

        name = job.__module__.split(".")[-1]

        print(f"Running {name}")

        job()

        print(f"✓ {name}")


def synchronize(source, destination):

    destination.mkdir(parents=True, exist_ok=True)

    copied = 0

    for csv in source.glob("*.csv"):

        shutil.copy2(csv, destination / csv.name)

        copied += 1

    return copied


def validate(folder):

    files = sorted(folder.glob("*.csv"))

    if not files:

        raise RuntimeError(f"No CSV files found in {folder}")

    return len(files)

### Execute

In [0]:
print("=" * 80)
print("SMART MANUFACTURING INTELLIGENCE PLATFORM")
print("Manufacturing Data Generation")
print("=" * 80)

print("\nCleaning previous datasets...")

clean_folder(MASTER_DATA)

clean_folder(TRANSACTIONAL_DATA)

print("✓ Previous datasets removed")

execute(

    MASTER_GENERATORS,

    "Generating Master Data"

)

execute(

    TRANSACTIONAL_GENERATORS,

    "Generating Transactional Data"

)

### Validate & Synchronize

In [0]:
master_files = validate(MASTER_DATA)

transaction_files = validate(TRANSACTIONAL_DATA)

master_copied = synchronize(

    MASTER_DATA,

    MASTER_VOLUME

)

transaction_copied = synchronize(

    TRANSACTIONAL_DATA,

    TRANSACTION_VOLUME

)

print("\n" + "=" * 80)
print("Generation Summary")
print("=" * 80)

print(f"Master datasets        : {master_files}")
print(f"Transactional datasets : {transaction_files}")
print(f"Files synchronized     : {master_copied + transaction_copied}")

print(f"\nDestination")
print(UNITY_VOLUME)

### Finish

In [0]:
print("\n" + "=" * 80)

print("SMIP v1.0 Pipeline Ready")

print("=" * 80)

print(f"Master datasets        : {master_files}")
print(f"Transactional datasets : {transaction_files}")
print(f"Files synchronized     : {master_copied + transaction_copied}")

print("\nNext Step")

print("Run Databricks Workflow → SMIP Production Pipeline")

print("=" * 80)